In [2]:
# import tensorflow as tf
# from tensorflow.examples.tutorials.mnist import input_data
# mnist = input_data.read_data_sets('MNIST_data', one_hot=True)
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # 强制 TF 2.x 以 TF 1.x 模式运行
import numpy as np

# 1. 使用现代的 Keras API 下载并加载 MNIST 数据集
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. 数据预处理：归一化并调整形状以匹配 placeholder 的维度
x_train = x_train.reshape(-1, 784).astype(np.float32) / 255.0
x_test = x_test.reshape(-1, 784).astype(np.float32) / 255.0

# 3. 对标签进行独热编码 (One-hot encode)
def one_hot_encode(labels, num_classes=10):
    return np.eye(num_classes)[labels]

y_train = one_hot_encode(y_train)
y_test = one_hot_encode(y_test)

# 4. 辅助函数：获取随机批次数据 (替代废弃的 mnist.train.next_batch)
def next_batch(batch_size, data, labels):
    idx = np.random.choice(len(data), batch_size, replace=False)
    return data[idx], labels[idx]

# ==========================================
# 下方为你原本的作业网络结构代码
# ==========================================

learning_rate = 1e-4
keep_prob_rate = 0.7 
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1.0})
    correct_prediction = tf.equal(tf.argmax(y_pre,1), tf.argmax(v_ys,1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1.0})
    return result

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# 定义输入占位符
xs = tf.placeholder(tf.float32, [None, 784]) 
ys = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 卷积层 1
W_conv1 = weight_variable([7, 7, 1, 32])      
b_conv1 = bias_variable([32])                    
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1) 
h_pool1 = max_pool_2x2(h_conv1)               

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])     
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2) 
h_pool2 = max_pool_2x2(h_conv2)               

# 全连接层 1
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

# 交叉熵函数
# 注意：加上 1e-10 防止 log(0) 导致 loss 变成 NaN
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction + 1e-10), reduction_indices=[1]))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    init = tf.global_variables_initializer()
    sess.run(init)
    
    for i in range(max_epoch):
        # 使用新的 next_batch 函数获取数据
        batch_xs, batch_ys = next_batch(100, x_train, y_train)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        
        if i % 100 == 0:
            print(f"Step {i}, test accuracy: ", compute_accuracy(x_test[:1000], y_test[:1000]))


Instructions for updating:
non-resource variables are not supported in the long term

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.

Step 0, test accuracy:  0.05
Step 100, test accuracy:  0.875
Step 200, test accuracy:  0.919
Step 300, test accuracy:  0.932
Step 400, test accuracy:  0.947
Step 500, test accuracy:  0.952
Step 600, test accuracy:  0.955
Step 700, test accuracy:  0.964
Step 800, test accuracy:  0.968
Step 900, test accuracy:  0.974
Step 1000, test accuracy:  0.975
Step 1100, test accuracy:  0.975
Step 1200, test accuracy:  0.977
Step 1300, test accuracy:  0.979
Step 1400, test accuracy:  0.978
Step 1500, test accuracy:  0.982
Step 1600, test accuracy:  0.981
Step 1700, test accuracy:  0.981
Step 1800, test accuracy:  0.98
Step 1900, test accuracy:  0.984
